# 14. Regularyzacja: Ridge, Lasso, Elastic Net oraz regresja logistyczna

Regularyzacja dodaje do funkcji kosztu karę za zbyt duże współczynniki.

Dla regresji liniowej:

$$
SSE(\beta)=\sum_i (y_i-x_i^T\beta)^2
$$

Ridge/L2:

$$
SSE(\beta)+\lambda\sum_{j>0}\beta_j^2
$$

Lasso/L1:

$$
SSE(\beta)+\lambda\sum_{j>0}|\beta_j|
$$

Elastic Net:

$$
SSE(\beta)+\lambda\left(\alpha\sum_{j>0}|\beta_j|+\frac{1-\alpha}{2}\sum_{j>0}\beta_j^2\right)
$$

W praktyce zwykle **nie karzemy interceptu**.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.precision', 6)

## 1. Ridge na przykładzie wielomianu: overfitting kontra stabilizacja

Zbudujemy wielomian dużego stopnia. Bez regularyzacji może świetnie dopasować trening, ale zachowywać się niestabilnie między punktami.

Ridge mówi: „dopasuj dane, ale nie pozwól współczynnikom urosnąć za bardzo”.

Rozwiązanie Ridge dla macierzy $X$:

$$
\hat\beta_{ridge}=(X^TX+\lambda P)^{-1}X^Ty
$$

Macierz $P$ ma zero na pozycji interceptu, jeśli nie karzemy interceptu.

In [2]:
rng = np.random.default_rng(123)
x = np.linspace(-3, 3, 36)
y_true = np.sin(x) + 0.25*x
y = y_true + rng.normal(0, 0.35, size=len(x))

perm = rng.permutation(len(x))
train_idx = perm[:24]
test_idx = perm[24:]

x_train, y_train = x[train_idx], y[train_idx]
x_test, y_test = x[test_idx], y[test_idx]

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x_train, y_train, label='train')
ax.scatter(x_test, y_test, label='test')
ax.plot(np.linspace(-3,3,300), np.sin(np.linspace(-3,3,300)) + 0.25*np.linspace(-3,3,300), label='true signal')
ax.set_title('Dane do pokazania overfittingu')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [3]:
def polynomial_features(x, degree):
    x = np.asarray(x)
    return np.column_stack([x**d for d in range(1, degree+1)])


def standardize_fit_transform(X_train, X_other=None):
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1.0
    X_train_s = (X_train - mean) / std
    if X_other is None:
        return X_train_s, mean, std
    return X_train_s, (X_other - mean) / std, mean, std


def add_intercept(X):
    return np.column_stack([np.ones(X.shape[0]), X])


def ridge_fit(X, y, lam):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    k = X.shape[1]
    P = np.eye(k)
    P[0, 0] = 0.0
    beta = np.linalg.pinv(X.T @ X + lam * P) @ X.T @ y
    return beta


def mse(y_true, y_pred):
    return float(np.mean((np.asarray(y_true)-np.asarray(y_pred))**2))

In [4]:
degree = 12
Xtr_raw = polynomial_features(x_train, degree)
Xte_raw = polynomial_features(x_test, degree)
Xgrid_raw = polynomial_features(np.linspace(-3, 3, 300), degree)
Xtr_s, Xte_s, mean_poly, std_poly = standardize_fit_transform(Xtr_raw, Xte_raw)
Xgrid_s = (Xgrid_raw - mean_poly) / std_poly

Xtr = add_intercept(Xtr_s)
Xte = add_intercept(Xte_s)
Xgrid = add_intercept(Xgrid_s)
x_grid = np.linspace(-3, 3, 300)

rows = []
fits_ridge = {}
for lam in [0.0, 0.1, 1.0, 10.0, 100.0]:
    beta = ridge_fit(Xtr, y_train, lam)
    fits_ridge[lam] = beta
    rows.append({
        'lambda': lam,
        'train_MSE': mse(y_train, Xtr @ beta),
        'test_MSE': mse(y_test, Xte @ beta),
        'coef_L2_norm_without_intercept': float(np.linalg.norm(beta[1:])),
    })

pd.DataFrame(rows)

,lambda,train_MSE,test_MSE,coef_L2_norm_without_intercept
0,0.0,0.069272,6.782784,381.000374
1,0.1,0.104843,0.085815,2.340495
2,1.0,0.158760,0.216566,1.498339
3,10.0,0.393546,0.431377,0.707030
4,100.0,0.891707,0.825233,0.201252


In [5]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_train, y_train, label='train')
ax.scatter(x_test, y_test, label='test')
for lam in [0.0, 0.1, 10.0, 100.0]:
    beta = fits_ridge[lam]
    ax.plot(x_grid, Xgrid @ beta, label=f'lambda={lam}')
ax.set_xlabel('x')
ax.set_ylabel('prediction')
ax.set_title('Ridge stabilizuje wielomian dużego stopnia')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 2. Lasso i Elastic Net przez soft-thresholding

Lasso może zerować współczynniki. Intuicyjnie kara L1 ma „ostre rogi”, które sprzyjają rozwiązaniom dokładnie na osiach.

Dla jednego kroku coordinate descent pojawia się operator soft-thresholding:

$$
S(z,\gamma)=sign(z)\max(|z|-\gamma,0)
$$

Dla Elastic Net aktualizacja ma postać:

$$
\beta_j \leftarrow \frac{S(\rho_j,\lambda\alpha)}{z_j + \lambda(1-\alpha)}
$$

gdzie $\alpha=1$ daje Lasso, $\alpha=0$ daje Ridge w wersji coordinate descent.

In [6]:
def soft_threshold(z, gamma):
    if z > gamma:
        return z - gamma
    if z < -gamma:
        return z + gamma
    return 0.0


def elastic_net_coordinate_descent(X, y, alpha=0.1, l1_ratio=1.0, max_iter=5000, tol=1e-8):
    # X powinno być wystandaryzowane, bez interceptu. Intercept = mean(y).
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    y_mean = y.mean()
    yc = y - y_mean
    beta = np.zeros(p)
    X_sq = (X**2).sum(axis=0) / n
    for _ in range(max_iter):
        beta_old = beta.copy()
        for j in range(p):
            residual = yc - X @ beta + X[:, j] * beta[j]
            rho = (X[:, j] @ residual) / n
            beta[j] = soft_threshold(rho, alpha*l1_ratio) / (X_sq[j] + alpha*(1-l1_ratio))
        if np.max(np.abs(beta - beta_old)) < tol:
            break
    return y_mean, beta

# Dane z korelacją między cechami.
rng = np.random.default_rng(202)
n = 40
x1 = rng.normal(0, 1, n)
x2 = x1 + rng.normal(0, 0.15, n)  # silnie skorelowana z x1
x3 = rng.normal(0, 1, n)
X_raw = np.column_stack([x1, x2, x3])
y_reg = 3.0*x1 + 0.0*x2 + 0.3*x3 + rng.normal(0, 0.6, n)
X_std, X_mean, X_stddev = standardize_fit_transform(X_raw)

In [7]:
rows = []
for l1_ratio in [0.0, 0.5, 1.0]:
    for alpha in [0.0, 0.02, 0.10, 0.30, 0.80]:
        intercept, beta = elastic_net_coordinate_descent(X_std, y_reg, alpha=alpha, l1_ratio=l1_ratio)
        pred = intercept + X_std @ beta
        rows.append({
            'l1_ratio': l1_ratio,
            'alpha_lambda': alpha,
            'intercept': intercept,
            'beta_x1': beta[0],
            'beta_x2_correlated_with_x1': beta[1],
            'beta_x3': beta[2],
            'MSE': mse(y_reg, pred),
            'nonzero_coefficients': int(np.sum(np.abs(beta) > 1e-8)),
        })

pd.DataFrame(rows)

,l1_ratio,alpha_lambda,intercept,beta_x1,beta_x2_correlated_with_x1,beta_x3,MSE,nonzero_coefficients
0,0.0,0.00,0.540062,4.779304,-1.559800,0.371583,0.372667,3
1,0.0,0.02,0.540062,2.508319,0.676503,0.356466,0.456444,3
2,0.0,0.10,0.540062,1.765573,1.289367,0.309009,0.539334,3
3,0.0,0.30,0.540062,1.470933,1.304044,0.223002,0.722151,3
4,0.0,0.80,0.540062,1.163576,1.100055,0.110417,1.420687,3
5,0.5,0.00,0.540062,4.779304,-1.559800,0.371583,0.372667,3
6,0.5,0.02,0.540062,3.015876,0.173959,0.351380,0.423403,3
7,0.5,0.10,0.540062,1.980535,1.095135,0.279779,0.517510,3
8,0.5,0.30,0.540062,1.569050,1.244338,0.127661,0.707007,3
9,0.5,0.80,0.540062,1.208414,1.082855,0.000000,1.401054,2


Warto pokazać studentom:

- Ridge zwykle zostawia wszystkie współczynniki niezerowe, ale je zmniejsza.
- Lasso może wyzerować część współczynników.
- Elastic Net łączy oba efekty i bywa wygodny przy skorelowanych cechach.

## 3. Regularyzacja w regresji logistycznej

Dla regresji logistycznej funkcja kosztu to $NLL$, a nie $SSE$.

Bez regularyzacji:

$$
NLL(\beta)=-\sum_i \left[y_i\eta_i - \log(1+e^{\eta_i})\right]
$$

Z L2:

$$
NLL_{ridge}(\beta)=NLL(\beta)+\frac{\lambda}{2}\sum_{j>0}\beta_j^2
$$

To pomaga szczególnie przy separacji i małych danych.

In [8]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -35, 35)
    return 1 / (1 + np.exp(-z))


def loglik_logistic(beta, X, y):
    eta = X @ beta
    return float(np.sum(y * eta - np.logaddexp(0, eta)))


def fit_logistic_irls_l2(X, y, l2=0.0, max_iter=100, tol=1e-10):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, k = X.shape
    beta = np.zeros(k)
    P = np.eye(k)
    P[0, 0] = 0.0
    converged = False
    for iteration in range(max_iter):
        eta = X @ beta
        p = sigmoid(eta)
        W = p * (1 - p)
        grad = X.T @ (y - p) - l2 * (P @ beta)
        info = (X.T * W) @ X + l2 * P
        step = np.linalg.pinv(info) @ grad
        beta_new = beta + step
        if np.max(np.abs(step)) < tol:
            beta = beta_new
            converged = True
            break
        beta = beta_new
    return {
        'beta': beta,
        'll_unpenalized': loglik_logistic(beta, X, y),
        'converged': converged,
        'iterations': iteration + 1,
    }

In [9]:
sep_x = np.array([-3, -2, -1, 1, 2, 3], dtype=float)
sep_y = np.array([0, 0, 0, 1, 1, 1], dtype=float)
X_sep = np.column_stack([np.ones_like(sep_x), sep_x])

rows = []
for lam in [0.0, 0.01, 0.1, 1.0, 10.0]:
    fit_sep = fit_logistic_irls_l2(X_sep, sep_y, l2=lam, max_iter=80)
    rows.append({
        'lambda_L2': lam,
        'intercept': fit_sep['beta'][0],
        'slope': fit_sep['beta'][1],
        'unpenalized_log_likelihood_at_solution': fit_sep['ll_unpenalized'],
        'converged': fit_sep['converged'],
        'iterations': fit_sep['iterations'],
    })

pd.DataFrame(rows)

,lambda_L2,intercept,slope,unpenalized_log_likelihood_at_solution,converged,iterations
0,0.00,1.671643e-01,52.782977,-1.410076e-23,False,80
1,0.01,-1.525848e-15,3.946162,-3.905013e-02,True,10
2,0.10,-1.001689e-16,2.294755,-2.143087e-01,True,8
3,1.00,-2.099906e-16,1.104404,-8.523952e-01,True,6
4,10.00,-1.056085e-17,0.363358,-2.424210e+00,True,4


In [10]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sep_x, sep_y, label='data')
x_grid = np.linspace(-3.5, 3.5, 300)
for lam in [0.01, 0.1, 1.0, 10.0]:
    fit_sep = fit_logistic_irls_l2(X_sep, sep_y, l2=lam, max_iter=80)
    ax.plot(x_grid, sigmoid(fit_sep['beta'][0] + fit_sep['beta'][1]*x_grid), label=f'lambda={lam}')
ax.set_xlabel('x')
ax.set_ylabel('P(y=1)')
ax.set_title('L2 w logistycznej: silniejsza kara = łagodniejsza krzywa')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 4. `C` w scikit-learn

W scikit-learn dla `LogisticRegression` parametr `C` jest odwrotnością siły regularyzacji:

$$
C \approx \frac{1}{\lambda}
$$

Dlatego:

- małe `C` oznacza silną regularyzację,
- duże `C` oznacza słabą regularyzację.

Poniższa komórka jest opcjonalna. Jeśli w środowisku nie ma scikit-learn, notebook nadal działa.

In [11]:
try:
    from sklearn.linear_model import LogisticRegression
    sklearn_available = True
except Exception as exc:
    sklearn_available = False
    print('scikit-learn unavailable:', exc)

if sklearn_available:
    rows = []
    for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
        clf = LogisticRegression(C=C, penalty='l2', solver='lbfgs')
        clf.fit(sep_x.reshape(-1, 1), sep_y)
        rows.append({
            'C': C,
            'intercept': clf.intercept_[0],
            'slope': clf.coef_[0, 0],
        })
    display(pd.DataFrame(rows))

,C,intercept,slope
0,0.01,1.041267e-18,0.056081
1,0.10,-3.752324e-18,0.363358
2,1.00,3.700345e-17,1.104404
3,10.00,1.757168e-17,2.294752
4,100.00,9.616439e-17,3.945926


## 5. Mini-ćwiczenia

### Ćwiczenie A

Dla jednego współczynnika $\beta=4$ i $\lambda=2$ kara Ridge wynosi:

$$
\lambda\beta^2=2\cdot 4^2=32
$$

Jeśli używamy wersji z $\frac{\lambda}{2}\beta^2$, kara wynosi $16$.

### Ćwiczenie B

Soft-thresholding:

$$
S(0.3,0.5)=0
$$

bo $|0.3|<0.5$.

### Ćwiczenie C

Jeśli w regresji logistycznej klasy są perfekcyjnie rozdzielone, zwykły model może mieć ogromne współczynniki. Ridge ogranicza ten efekt.